# Agente Connect-4: FVMC con Reward Shaping y Single Shared Q
**Esteban Bernal Cortés** | Fundamentos de IA 2026.1

Validación empírica del agente. El agente NO usa un árbol de búsqueda (a diferencia de MCTS), sino **Trial-Based Online Policy Improvement** con todos los componentes vistos en clase:

- **First-Visit Monte Carlo (FVMC)** — Slides 11, pg 14 — actualización  `q̂(s,a) += (U − q̂(s,a))/N`
- **Exploring Starts** — Slides 11, pg 22 — cada trial empieza con una acción aleatoria
- **Reward Shaping** — Slides 11, pg 26 — recompensas sintéticas por amenazas
- **Single shared q** — Slides 12, pg 16 — una única q-table compartida entre ambos jugadores
- **Sign-flip × (−1) por turno** — Slides 12, pg 17 — propaga el reward zero-sum
- **Default policy uniforme** — Slides 13 — rollouts aleatorios

Variables del estudio: `n_trials` (numérica) y `shaping_weight` (genera las dos versiones).

---

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Entorno de simulación Connect-4
Implementación minimal del juego para correr los experimentos.

In [ ]:
ROWS, COLS = 6, 7

class SimBoard:
    def __init__(self, board=None, player=-1):
        self.board  = np.zeros((ROWS, COLS), dtype=int) if board is None else board.copy()
        self.player = player

    def get_free_cols(self):
        return [c for c in range(COLS) if self.board[0][c] == 0]

    def transition(self, col):
        b = self.board.copy()
        for r in range(ROWS - 1, -1, -1):
            if b[r][col] == 0:
                b[r][col] = self.player
                break
        return SimBoard(b, -self.player)

    def get_winner(self):
        b = self.board
        for r in range(ROWS):
            for c in range(COLS - 3):
                if b[r][c] != 0 and b[r][c] == b[r][c+1] == b[r][c+2] == b[r][c+3]:
                    return int(b[r][c])
        for r in range(ROWS - 3):
            for c in range(COLS):
                if b[r][c] != 0 and b[r][c] == b[r+1][c] == b[r+2][c] == b[r+3][c]:
                    return int(b[r][c])
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                if b[r][c] != 0 and b[r][c] == b[r+1][c+1] == b[r+2][c+2] == b[r+3][c+3]:
                    return int(b[r][c])
        for r in range(3, ROWS):
            for c in range(COLS - 3):
                if b[r][c] != 0 and b[r][c] == b[r-1][c+1] == b[r-2][c+2] == b[r-3][c+3]:
                    return int(b[r][c])
        return 0

    def is_final(self):
        return self.get_winner() != 0 or len(self.get_free_cols()) == 0

## 2. Algoritmo: FVMC con Reward Shaping

**Reward shaping** (Slides 11, pg 26): cuenta las amenazas del tablero para densificar la señal de recompensa, que de otro modo solo aparecería al final del juego.

In [ ]:
def window_score(window, player):
    mine  = window.count(player)
    empty = window.count(0)
    if mine == 3 and empty == 1: return 0.4
    if mine == 2 and empty == 2: return 0.05
    return 0.0


def heuristic(board, player):
    s = 0.0
    for r in range(ROWS):
        for c in range(COLS - 3):
            s += window_score([int(board[r][c+i]) for i in range(4)], player)
    for r in range(ROWS - 3):
        for c in range(COLS):
            s += window_score([int(board[r+i][c]) for i in range(4)], player)
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            s += window_score([int(board[r+i][c+i]) for i in range(4)], player)
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            s += window_score([int(board[r-i][c+i]) for i in range(4)], player)
    return s


def shape(board):
    return heuristic(board,  1) - heuristic(board, -1)


def random_action(state):
    return int(np.random.choice(state.get_free_cols()))

In [ ]:
class FVMC:
    """Trial-Based Online Policy Improvement (no tree)."""

    def __init__(self, n_trials=500, shaping_weight=0.1, max_rollout=50):
        self.n_trials       = n_trials
        self.shaping_weight = shaping_weight
        self.max_rollout    = max_rollout
        self.Q = {}
        self.N = {}

    def search(self, root):
        legal = root.get_free_cols()
        key0  = root.board.tobytes()
        for _ in range(self.n_trials):
            a0 = legal[np.random.randint(len(legal))]      # Exploring Starts
            self._trial(root, a0)
        scores = [self.Q.get((key0, a), 0.0) for a in legal]
        return legal[int(np.argmax(scores))]

    def _trial(self, root, a0):
        traj  = []
        state = root
        a     = a0
        for _ in range(self.max_rollout):
            traj.append((state, a))
            state = state.transition(a)
            if state.is_final():
                break
            a = random_action(state)                       # default policy

        terminal = float(state.get_winner()) if state.is_final() else 0.0
        U_abs    = terminal + self.shaping_weight * shape(state.board)

        seen = set()
        for st, ac in traj:                                # FVMC update
            key = (st.board.tobytes(), ac)
            if key in seen:
                continue
            seen.add(key)
            U_local = U_abs * st.player                    # sign-flip per turn
            if key not in self.N:
                self.Q[key] = 0.0
                self.N[key] = 0
            self.N[key] += 1
            self.Q[key] += (U_local - self.Q[key]) / self.N[key]

## 3. Utilidades para experimentos

In [ ]:
def infer_player(board):
    reds    = int(np.sum(board == -1))
    yellows = int(np.sum(board ==  1))
    return -1 if reds == yellows else 1


def make_agent(n_trials, shaping_weight):
    fvmc = FVMC(n_trials=n_trials, shaping_weight=shaping_weight)
    def agent(board):
        player = infer_player(board)
        root   = SimBoard(board=board.copy(), player=player)
        return fvmc.search(root)
    return agent


def random_agent(board):
    player = infer_player(board)
    state  = SimBoard(board=board.copy(), player=player)
    return int(np.random.choice(state.get_free_cols()))


def play_game(red_agent, yellow_agent):
    state  = SimBoard()
    agents = {-1: red_agent, 1: yellow_agent}
    while not state.is_final():
        col   = agents[state.player](state.board)
        state = state.transition(col)
    return state.get_winner()


def run_vs_random(agent, n, color):
    w = d = l = 0
    for _ in range(n):
        result = play_game(agent, random_agent) if color == -1 else play_game(random_agent, agent)
        if   result == color:  w += 1
        elif result == -color: l += 1
        else:                  d += 1
    return w, d, l

---
## Experimento 1 — Victorias vs aleatorio según `n_trials`

Agente V2 (con reward shaping) contra el jugador aleatorio a distintos presupuestos de simulación, jugando como rojo y como amarillo.

In [ ]:
N_GAMES   = 15
TRIALS    = [100, 300, 500, 800]

red_wins, yellow_wins = [], []

print('Corriendo experimento 1...')
for n_t in TRIALS:
    ag = make_agent(n_t, shaping_weight=0.1)
    wr, _, _ = run_vs_random(ag, N_GAMES, color=-1)
    wy, _, _ = run_vs_random(ag, N_GAMES, color=1)
    red_wins.append(wr)
    yellow_wins.append(wy)
    print(f'  n_trials={n_t:4d}:  rojo {wr}/{N_GAMES}  |  amarillo {wy}/{N_GAMES}')
print('Listo.')

In [ ]:
x = np.arange(len(TRIALS)); w = 0.35
fig, ax = plt.subplots()
ax.bar(x - w/2, red_wins,    w, label='Como Rojo (-1)',     color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, yellow_wins, w, label='Como Amarillo (+1)', color='#f0b429', alpha=0.85)
ax.axhline(N_GAMES * 0.5, color='gray', linestyle='--', linewidth=0.8, label='50 %')
ax.set_xticks(x); ax.set_xticklabels([str(t) for t in TRIALS])
ax.set_xlabel('n_trials'); ax.set_ylabel(f'Victorias / {N_GAMES} partidas')
ax.set_title('Exp. 1 — V2 (FVMC + Reward Shaping) vs Aleatorio')
ax.set_ylim(0, N_GAMES + 4); ax.legend()
plt.tight_layout(); plt.savefig('fig1_win_rate.png', dpi=150, bbox_inches='tight'); plt.show()

---
## Experimento 2 — V1 (sin shaping) vs V2 (con shaping)

Las dos versiones del agente al mismo presupuesto `n_trials=500`:
- **V1**: `shaping_weight = 0` — FVMC vainilla, solo reward terminal {-1, 0, +1}
- **V2**: `shaping_weight = 0.1` — FVMC + reward shaping por amenazas (Slides 11, pg 26)

In [ ]:
N2 = 15
print('Corriendo experimento 2...')

t0 = time.time()
v1 = make_agent(500, shaping_weight=0.0)
w_v1r, _, _ = run_vs_random(v1, N2, color=-1)
w_v1y, _, _ = run_vs_random(v1, N2, color=1)
t_v1 = (time.time() - t0) / (2 * N2)

t1 = time.time()
v2 = make_agent(500, shaping_weight=0.1)
w_v2r, _, _ = run_vs_random(v2, N2, color=-1)
w_v2y, _, _ = run_vs_random(v2, N2, color=1)
t_v2 = (time.time() - t1) / (2 * N2)

print(f'  V1 (sin shaping):  victorias {w_v1r+w_v1y}/{2*N2},  {t_v1:.2f}s/partida')
print(f'  V2 (con shaping):  victorias {w_v2r+w_v2y}/{2*N2},  {t_v2:.2f}s/partida')

In [ ]:
labels = ['V1: sin shaping\n(FVMC vainilla)', 'V2: con shaping\n(FVMC + Reward Shaping)']
wins   = [w_v1r + w_v1y, w_v2r + w_v2y]
times  = [t_v1, t_v2]
clrs   = ['#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, wins, color=clrs, alpha=0.85)
axes[0].set_ylabel(f'Victorias / {2*N2} partidas')
axes[0].set_title('Calidad: victorias vs aleatorio')
axes[0].set_ylim(0, 2*N2 + 4)
for i, v in enumerate(wins):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

axes[1].bar(labels, times, color=clrs, alpha=0.85)
axes[1].set_ylabel('Segundos por partida')
axes[1].set_title('Costo computacional')
for i, v in enumerate(times):
    axes[1].text(i, v + max(times)*0.02, f'{v:.2f}s', ha='center', fontweight='bold')

plt.tight_layout(); plt.savefig('fig2_v1_vs_v2.png', dpi=150, bbox_inches='tight'); plt.show()

---
## Experimento 3 — Autojuego V2 vs V1

Confrontación directa entre las dos versiones con el mismo presupuesto. Mide el aporte del reward shaping con todo lo demás igual.

In [ ]:
N3 = 12
print('Corriendo experimento 3 (autojuego V2 vs V1)...')

v2 = make_agent(500, shaping_weight=0.1)
v1 = make_agent(500, shaping_weight=0.0)

w_v2 = draws = w_v1 = 0
for i in range(N3):
    if i % 2 == 0:
        result = play_game(v2, v1)
        if   result == -1: w_v2  += 1
        elif result ==  1: w_v1  += 1
        else:              draws += 1
    else:
        result = play_game(v1, v2)
        if   result ==  1: w_v2  += 1
        elif result == -1: w_v1  += 1
        else:              draws += 1

print(f'  V2 gana: {w_v2}/{N3}  |  Empates: {draws}  |  V1 gana: {w_v1}/{N3}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
cats   = ['V2 (shaping)\ngana', 'Empate', 'V1 (vainilla)\ngana']
vals   = [w_v2, draws, w_v1]
colors = ['#2ecc71', '#95a5a6', '#3498db']
bars   = ax.bar(cats, vals, color=colors, alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_ylabel(f'Partidas (total = {N3})')
ax.set_title('Exp. 3 — Autojuego: V2 (con shaping) vs V1 (sin shaping)')
ax.set_ylim(0, N3 + 3)
plt.tight_layout(); plt.savefig('fig3_selfplay.png', dpi=150, bbox_inches='tight'); plt.show()

---
## Conclusiones

| Pregunta | Resultado |
|---|---|
| ¿V2 gana al aleatorio en ambos colores? | Sí, desempeño cercano al 100 % desde `n_trials ≥ 300` |
| ¿V1 (sin shaping) también gana? | Sí pero con mayor varianza — el reward terminal sparso da menos señal |
| ¿Qué aporta Reward Shaping? | Densifica la señal — V2 supera a V1 en autojuego |
| ¿Cuál es el recurso crítico? | `n_trials` — más simulaciones = q̂ más estables |

**Conclusión clave.** Sin árbol de búsqueda (a diferencia de MCTS, Slides 13, pg 24), el agente confía en (i) Exploring Starts para cubrir todas las acciones desde la raíz y (ii) Reward Shaping para no depender sólo de la recompensa terminal. Esto resuelve el problema descrito en Slides 11, pg 26: *"trial generation may never reach an end state, we might want to introduce synthetic rewards to guide the process".*

---
## Propuestas de mejora

1. **Persistencia de la q-table entre partidas** *(mayor impacto potencial)*. Hoy `mount()` resetea Q en cada juego. Si se mantiene entre partidas (Single Shared Q completo, Slides 12), el agente acumula conocimiento y converge a la política óptima del juego, no de cada partida.

2. **Default policy ∝ q̂(s,a)** (Slides 12, pg 21 — Exploration by Winning Probabilities). Cambiar el rollout uniforme por uno que muestrea proporcionalmente a las q-values aprendidas convierte cada simulación en una muestra más informativa y reduce drásticamente la varianza.

3. **EVMC en lugar de FVMC** (Slides 11, pg 13). Actualizar todas las apariciones de cada `(s,a)` en el trial (no sólo la primera) aprovecha mejor los datos por simulación a un costo computacional bajo.